# ArgLLM baseline (DF-QuAD) — GPT-OSS 120B on COM2SENSE (500 claims, earlier sample from the training split)

This is the **earlier adaptation** of the ArgLLM prompts described in Section 5.1 of the paper: the model writes its own facts, and the argument prompt asks for one argument on each side instead of allowing the model to decline. Strengths are computed with the authors' DF-QuAD implementation (Uncertainpy). Results: 0.5 Base 52.2, Est. Base 73.8.

Run on Kaggle in batches of 100; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`).

In [5]:
!pip install -q openai datasets scikit-learn
import os, re, json, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/gptoss_com2sense_dfquad"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "openai/gpt-oss-120b"

ds = load_dataset("tasksource/com2sense", split="train")
df = ds.to_pandas()
df["gold"] = df["label"].apply(lambda x: 1 if str(x).strip().lower()=="true" else 0)
df["claim"] = df["sent"].astype(str)
true_ex = df[df["gold"]==1].sample(250, random_state=42)
false_ex = df[df["gold"]==0].sample(250, random_state=42)
baf_df = pd.concat([true_ex, false_ex]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("Loaded COM2SENSE:", len(baf_df), "| First:", baf_df.iloc[0]["claim"][:70])

def generate_facts(claim):
    prompt = f'For the claim below, produce 3-5 SHORT atomic factual statements, one per line. General commonsense, not a conclusion.\nClaim: "{claim}"'
    for _ in range(5):
        try:
            r=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. Never conclude."},{"role":"user","content":prompt}],
                temperature=0.3, max_tokens=1200)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(5); continue
            return ""
    return ""
print("✅ CELL 1 ready")

Loaded COM2SENSE: 500 | First: Tim was happy that November was coming up since he'd have a lot of tim
✅ CELL 1 ready


In [8]:
import os, subprocess, sys
r = subprocess.run(["git","clone","https://github.com/CLArg-group/argumentative-llms.git",
                    "/kaggle/working/argumentative-llms"], capture_output=True, text=True)
print("clone return code:", r.returncode)
print("stderr:", r.stderr[:200])

# find and add uncertainpy path
base = "/kaggle/working/argumentative-llms"
print("repo exists now:", os.path.exists(base))
for root, dirs, files in os.walk(base):
    if "uncertainpy" in dirs:
        print("found uncertainpy at:", root)
        sys.path.insert(0, root)
        break

clone return code: 0
stderr: Cloning into '/kaggle/working/argumentative-llms'...

repo exists now: True
found uncertainpy at: /kaggle/working/argumentative-llms/Uncertainpy/src


In [11]:
import sys, importlib
p = "/kaggle/working/argumentative-llms/Uncertainpy/src"
if p not in sys.path:
    sys.path.insert(0, p)
print("path in sys.path:", p in sys.path)
print("first 3 sys.path:", sys.path[:3])

# clear any cached failed import
for mod in list(sys.modules.keys()):
    if "uncertainpy" in mod:
        del sys.modules[mod]

importlib.invalidate_caches()

from uncertainpy.gradual import BAG, Argument, computeStrengthValues, ProductAggregation, LinearInfluence
agg_f = ProductAggregation(); inf_f = LinearInfluence(conservativeness=1)
print("✅ imports ready")

path in sys.path: True
first 3 sys.path: ['/kaggle/working/argumentative-llms/Uncertainpy/src', '/kaggle/working/argumentative-llms/Uncertainpy/src', '/kaggle/working/argumentative-llms/Uncertainpy/src']
✅ imports ready


In [13]:
import re, time
def llm_raw(prompt, max_tokens=1500, temperature=0.0):
    for _ in range(5):
        try:
            r=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"user","content":prompt}], temperature=temperature, max_tokens=max_tokens)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(5); continue
            return ""
    return ""

def gen_argument(claim, facts, supporting):
    stance = "SUPPORTS (argues TRUE)" if supporting else "ATTACKS (argues FALSE)"
    prompt = f'Claim: "{claim}"\nFacts: {facts}\nWrite ONE short argument (under 30 words) that {stance} the claim. Output only the argument, no label.'
    txt = llm_raw(prompt, 1200, 0.2)
    return txt if txt else "N/A"

def get_lik(text, claim):
    prompt = f'On a scale 0.0 to 1.0, how strongly does this argument hold for the claim?\nClaim: "{claim}"\nArgument: "{text}"\nOutput only a number between 0 and 1.'
    txt = llm_raw(prompt, 800, 0.0)
    m = re.search(r"([01](?:\.\d+)?)", txt or "")
    if m:
        try:
            return max(0.0, min(1.0, float(m.group(1))))
        except:
            return 0.5
    return 0.5

def arg_base(text, claim, supporting):
    if text == "N/A" or not text:
        return None
    return get_lik(text, claim)

def topic_base(claim, facts):
    prompt = f'On a scale 0.0 to 1.0, how likely is this claim TRUE?\nClaim: "{claim}"\nFacts: {facts}\nOutput only a number between 0 and 1.'
    txt = llm_raw(prompt, 800, 0.0)
    m = re.search(r"([01](?:\.\d+)?)", txt or "")
    if m:
        try:
            return max(0.0, min(1.0, float(m.group(1))))
        except:
            return 0.5
    return 0.5

print("✅ helpers ready")

✅ helpers ready


In [14]:
DFQ_CKPT = WORK_DIR / "gptoss_com2sense_dfquad_ckpt.pkl"
def load_dfq():
    return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d):
    pickle.dump(d, open(DFQ_CKPT,"wb"))
def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (GPT-OSS DF-QuAD)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]; claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True)
            att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts, "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True), "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done")
print("✅ runner ready")

✅ runner ready


The Kaggle session was restarted after the first 100 claims; the next two cells reload the saved claims from a Kaggle dataset before the run continues.

In [15]:
import glob
hits = glob.glob("/kaggle/input/**/*dfquad*.pkl", recursive=True)
print("found:", hits)

found: ['/kaggle/input/gptoss-com2sense-dfquad-100-backup/gptoss_com2sense_dfquad_100.pkl']


In [16]:
path = "/kaggle/input/gptoss-com2sense-dfquad-100-backup/gptoss_com2sense_dfquad_100.pkl"
data = pickle.load(open(path, "rb"))
print("loaded:", len(data), "items")

restored = {}
unmatched = 0
for r in data:
    idx = baf_df.index[baf_df["claim"] == r["claim"]]
    if len(idx):
        restored[int(idx[0])] = r
    else:
        unmatched += 1
save_dfq(restored)
print(f"Restored {len(load_dfq())} examples | unmatched: {unmatched}")

loaded: 100 items
Restored 100 examples | unmatched: 0


In [7]:
DFQ_CKPT = WORK_DIR / "gptoss_com2sense_dfquad_ckpt.pkl"
def load_dfq(): return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d): pickle.dump(d, open(DFQ_CKPT,"wb"))
def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (GPT-OSS DF-QuAD)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]; claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True); att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts, "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True), "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo): print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done\n")

run_dfquad_batch(100)

# EVAL
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]
print(f"=== GPT-OSS COM2SENSE DF-QuAD ({len(idxs)}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 0/500. Running 100 (GPT-OSS DF-QuAD)...
  10/100 → 10 total
  20/100 → 20 total
  30/100 → 30 total
  40/100 → 40 total
  50/100 → 50 total
  60/100 → 60 total
  70/100 → 70 total
  80/100 → 80 total
  90/100 → 90 total
  100/100 → 100 total
✅ 100/500 done

=== GPT-OSS COM2SENSE DF-QuAD (100) ===
  DF-QuAD 0.5 Base (fair): 47.0%  (F1 62.9)
  DF-QuAD Est. Base: 74.0%  (F1 77.2)


Backed up 100


In [19]:
# RUN
run_dfquad_batch(100)

# ACCURACY
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]
print(f"\n=== GPT-OSS COM2SENSE DF-QuAD ({len(idxs)}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 100/500. Running 100 (GPT-OSS DF-QuAD)...
  10/100 → 110 total
  20/100 → 120 total
  30/100 → 130 total
  40/100 → 140 total
  50/100 → 150 total
  60/100 → 160 total
  70/100 → 170 total
  80/100 → 180 total
  90/100 → 190 total
  100/100 → 200 total
✅ 200/500 done

=== GPT-OSS COM2SENSE DF-QuAD (200) ===
  DF-QuAD 0.5 Base (fair): 55.0%  (F1 69.6)  TN=7 FP=88 FN=2 TP=103
  DF-QuAD Est. Base: 76.0%  (F1 80.3)  TN=54 FP=41 FN=7 TP=98


Backed up 200


In [20]:
# RUN
run_dfquad_batch(100)

# ACCURACY
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]
print(f"\n=== GPT-OSS COM2SENSE DF-QuAD ({len(idxs)}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 200/500. Running 100 (GPT-OSS DF-QuAD)...
  10/100 → 210 total
  20/100 → 220 total
  30/100 → 230 total
  40/100 → 240 total
  50/100 → 250 total
  60/100 → 260 total
  70/100 → 270 total
  80/100 → 280 total
  90/100 → 290 total
  100/100 → 300 total
✅ 300/500 done

=== GPT-OSS COM2SENSE DF-QuAD (300) ===
  DF-QuAD 0.5 Base (fair): 52.0%  (F1 66.7)  TN=12 FP=138 FN=6 TP=144
  DF-QuAD Est. Base: 72.0%  (F1 77.0)  TN=75 FP=75 FN=9 TP=141


Backed up 300


In [21]:
# RUN
run_dfquad_batch(100)

# ACCURACY
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]
print(f"\n=== GPT-OSS COM2SENSE DF-QuAD ({len(idxs)}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 300/500. Running 100 (GPT-OSS DF-QuAD)...
  10/100 → 310 total
  20/100 → 320 total
  30/100 → 330 total
  40/100 → 340 total
  50/100 → 350 total
  60/100 → 360 total
  70/100 → 370 total
  80/100 → 380 total
  90/100 → 390 total
  100/100 → 400 total
✅ 400/500 done

=== GPT-OSS COM2SENSE DF-QuAD (400) ===
  DF-QuAD 0.5 Base (fair): 52.8%  (F1 67.4)  TN=16 FP=179 FN=10 TP=195
  DF-QuAD Est. Base: 73.8%  (F1 78.7)  TN=101 FP=94 FN=11 TP=194


Backed up 400


In [22]:
# RUN final 100
run_dfquad_batch(100)

# FINAL ACCURACY
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]
print(f"\n=== GPT-OSS COM2SENSE DF-QuAD FINAL ({len(idxs)}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 400/500. Running 100 (GPT-OSS DF-QuAD)...
  10/100 → 410 total
  20/100 → 420 total
  30/100 → 430 total
  40/100 → 440 total
  50/100 → 450 total
  60/100 → 460 total
  70/100 → 470 total
  80/100 → 480 total
  90/100 → 490 total
  100/100 → 500 total
✅ 500/500 done

=== GPT-OSS COM2SENSE DF-QuAD FINAL (500) ===
  DF-QuAD 0.5 Base (fair): 52.2%  (F1 66.4)  TN=25 FP=225 FN=14 TP=236
  DF-QuAD Est. Base: 73.8%  (F1 78.3)  TN=132 FP=118 FN=13 TP=237


Backed up 500
